# IntAct Unsupervised Pre-training - Data Preparation

Using [IntAct dataset](https://www.ebi.ac.uk/intact/download/datasets#mutations) for unsupervised pre-training of StaBddG's folding part.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

from stabddg.constants import ALPHABET
from stabddg.intact import (
    IntactDataController,
    fetch_alphafold_atoms_parallel,
    fetch_assemblies_atoms_parallel,
    fetch_assemblies_for_uniprots_parallel,
    normalize_entry,
    fetch_alphafold_atoms_df
)
from stabddg.intact_pretrain import IntactDataset
from stabddg.utils.gcp import download_dir_from_gcp, upload_dir_to_gcp

In [ ]:
pd.set_option('display.max_columns', None)

# Defines

In [ ]:
data_dir = "data"

In [ ]:
proteins_dir = os.path.join(data_dir, "intact", "proteins")
assemblies_dir = os.path.join(data_dir, "intact", "assemblies")

os.makedirs(proteins_dir, exist_ok=True)
os.makedirs(assemblies_dir, exist_ok=True)

In [ ]:
gcp_output_bucket = "stab-ddg-unsup"
gcp_output_prefix = "data/intact"

# Raw Data

## IntAct Mutations dataset

IntAct Mutations dataset contains info on pairs of proteins: how a mutation in one of the protein impacts the binding of the two proteins.

The dataset does not contain any ∆G measurements per se.

In [ ]:
dc_intact = IntactDataController(
    output_dir=os.path.join(os.path.join(data_dir, "intact"))
)

In [ ]:
dc_intact.prepare_data()

# Protein 3D Structures

AlphaFold only for the time being

In [ ]:
df_mutations = pd.read_parquet(os.path.join(data_dir, "intact", "df_intact_mutations.parquet"))

In [ ]:
cur_parquet_dir = os.path.join(proteins_dir, "parquet")
cur_safetensors_dir = os.path.join(proteins_dir, "safetensors")

os.makedirs(cur_parquet_dir, exist_ok=True)
os.makedirs(cur_safetensors_dir, exist_ok=True)

In [ ]:
# Collecting all UniProt codes
uniprot_codes = pd.concat(
    [
        df_mutations["participant_protein"],
        df_mutations["affected_protein_ac"]
    ],
    ignore_index=True
).unique().tolist()

In [ ]:
step_outcome = fetch_alphafold_atoms_parallel(
    uniprot_codes=uniprot_codes,
    parquet_dir=cur_parquet_dir,
    safetensors_dir=cur_safetensors_dir,
    max_workers=128
)

In [ ]:
step_outcome_success = {
    k: v
    for k, v in step_outcome.items() if v["status"] == "success"
}

uniprot_codes_success = list(step_outcome_success.keys())

df_mutations_filtered = df_mutations.loc[
    (df_mutations["participant_protein"].isin(uniprot_codes_success))
    & (df_mutations["affected_protein_ac"].isin(uniprot_codes_success))
]
df_mutations_filtered = df_mutations_filtered.reset_index(drop=True)

In [ ]:
df_mutations_filtered.to_parquet(os.path.join(data_dir, "intact", "df_intact_mutations_filtered.parquet"))

# Assemblies

In [ ]:
cur_parquet_dir = os.path.join(assemblies_dir, "parquet")
cur_safetensors_dir = os.path.join(assemblies_dir, "safetensors")

os.makedirs(cur_parquet_dir, exist_ok=True)
os.makedirs(cur_safetensors_dir, exist_ok=True)

## Getting All Assemblies

In [ ]:
df_mutations_filtered = pd.read_parquet(os.path.join(data_dir, "intact", "df_intact_mutations_filtered.parquet"))

In [ ]:
uniprots_pairs = [
    list(k) 
    for k, v in df_mutations_filtered.groupby(['participant_protein', 'affected_protein_ac']).groups.items()
]

In [ ]:
df_assemblies_raw = fetch_assemblies_for_uniprots_parallel(
    uniprots_pairs=uniprots_pairs,
    max_workers=32
)

In [ ]:
df_assemblies_raw.drop(columns=["mutations"]).to_parquet(os.path.join(data_dir, "intact", "df_assemblies_raw.parquet"))

## Getting All Structures + Writing as Safetensors

In [ ]:
df_assemblies_raw = pd.read_parquet(
    os.path.join(data_dir, "intact", "df_assemblies_raw.parquet")
)

df_assemblies = df_assemblies_raw.copy()

df_assemblies = df_assemblies.loc[
    df_assemblies["biological_assembly"].notna()
].reset_index(drop=True)

df_assemblies = df_assemblies.loc[
    df_assemblies["score"] == df_assemblies.groupby("pair_idx")["score"].transform("min")
].reset_index(drop=True)

df_assemblies["entry_id"] = df_assemblies["biological_assembly"].apply(normalize_entry)

In [ ]:
df_assemblies.to_parquet(
    os.path.join(data_dir, "intact", "df_assemblies.parquet")
)

In [ ]:
assembly_ids = df_assemblies["biological_assembly"].unique().tolist()

In [ ]:
step_outcome = fetch_assemblies_atoms_parallel(
    asm_ids=assembly_ids,
    parquet_dir=cur_parquet_dir,
    safetensors_dir=cur_safetensors_dir,
    max_workers=128
)

In [ ]:
step_outcome_success = {
    k: v
    for k, v in step_outcome.items() if v["status"] == "success"
}

assembly_ids_success = list(step_outcome_success.keys())

df_assemblies_filtered = df_assemblies.loc[
    df_assemblies["biological_assembly"].isin(assembly_ids_success)
]
df_assemblies_filtered = df_assemblies_filtered.reset_index(drop=True)

In [ ]:
# Adding data about the total length of the complex
dict_lengths = {}
for k, v in tqdm(step_outcome_success.items()):
    df_cur = pd.read_parquet(v['parquet_path'])
    dict_lengths[k] = df_cur.groupby(["chain", "resnum_label"]).ngroups

df_dict_lengths = pd.DataFrame.from_dict(
    dict_lengths, 
    orient='index',
    columns=['biological_assembly_length']
)

In [ ]:
df_assemblies_filtered = df_assemblies_filtered.merge(
    df_dict_lengths,
    left_on="biological_assembly",
    right_index=True
)

In [ ]:
df_assemblies_filtered.to_parquet(os.path.join(data_dir, "intact", "df_assemblies_filtered.parquet"))

# Uploading to GCP

In [ ]:
upload_dir_to_gcp(
    bucket_name=gcp_output_bucket,
    local_dir="data/intact/",
    dst_prefix=gcp_output_prefix,
    region="europe-west4",
)

# Downloading from GCP

In [ ]:
download_dir_from_gcp(
    bucket_name=gcp_output_bucket,
    src_prefix=gcp_output_prefix,
    local_dir="data/intact/",
)

# Train / Valid / Test split

In [ ]:
valid_size = 0.1
test_size = 0.1
random_state = 42

In [ ]:
feature_type_pos = [
    "mutation causing(MI:2227)",
    "mutation increasing(MI:0382)",
    "mutation increasing rate(MI:1131)",
    "mutation increasing strength(MI:1132)"
]
feature_type_neg = [
    "mutation decreasing(MI:0119)",
    "mutation decreasing rate(MI:1130)",
    "mutation decreasing strength(MI:1133)",
    "mutation disrupting(MI:0573)",
    "mutation disrupting rate(MI:1129)",
    "mutation disrupting strength(MI:1128)",
]

In [ ]:
df_mutations = pd.read_parquet(os.path.join(data_dir, "intact", "df_intact_mutations_filtered.parquet"))
df_assemblies = pd.read_parquet(os.path.join(data_dir, "intact", "df_assemblies_filtered.parquet"))

In [ ]:
# Only keeping mutations of the same length as the originals (in
# line with https://arxiv.org/html/2507.05502)
df_mutations = df_mutations.loc[
    (df_mutations["resulting_sequence"] != ".")
    &(df_mutations["original_sequence"].str.len() == df_mutations["resulting_sequence"].str.len())
].reset_index(drop=True)

In [ ]:
# First we need to merge mutations and assemblies
df = df_mutations.merge(
    right=df_assemblies,
    on=["participant_protein", "affected_protein_ac"],
    how="inner",
    validate="m:m"
)

In [ ]:
# Expressing 'resulting_sequence' as sequence codes
df["affected_protein_ac_seq_mut"] = df["affected_protein_ac_seq_mut"].str.replace(r".", "")

# Calculating lengths
df["participant_protein_len"] = df["participant_protein_seq"].str.len()
df["affected_protein_ac_len"] = df["affected_protein_ac_seq"].str.len()
df["affected_protein_ac_mut_len"] = df["affected_protein_ac_seq_mut"].str.len()
# df["max_len"] = df[[
#     "biological_assembly_length",
#     "participant_protein_len",
#     "affected_protein_ac_len",
#     "affected_protein_ac_mut_len"
# ]].max(axis=1)
df["max_len"] = df["biological_assembly_length"]

# Converting range indices back to 1-based - making them aligned with the 
# residue_idx tensors
df["feature_ranges_start"] += 1
df["feature_ranges_end"] += 1

In [ ]:
# `feature_type_category` column is used for stratification
df["feature_type_category"] = "neutral"
df.loc[df["feature_type"].isin(feature_type_pos), "feature_type_category"] = "positive"
df.loc[df["feature_type"].isin(feature_type_neg), "feature_type_category"] = "negative"

In [ ]:
df_train, df_valid = train_test_split(
    df,
    test_size=(valid_size + test_size),
    stratify=df["feature_type_category"],
    random_state=random_state
)

In [ ]:
df_valid, df_test = train_test_split(
    df_valid,
    test_size=test_size / (valid_size + test_size),
    stratify=df_valid["feature_type_category"],
    random_state=random_state
)

In [ ]:
# Removing the `feature_type_category` column
for df_cur in [df_train, df_valid, df_test]:
    df_cur.drop(columns=["feature_type_category"], inplace=True)
    df_cur.reset_index(drop=True, inplace=True)

In [ ]:
df_train.to_parquet(os.path.join(data_dir, "intact", "df_intact_mutations_filtered_train.parquet"))
df_valid.to_parquet(os.path.join(data_dir, "intact", "df_intact_mutations_filtered_valid.parquet"))
df_test.to_parquet(os.path.join(data_dir, "intact", "df_intact_mutations_filtered_test.parquet"))